# 3.7 Autoencoder Anomaly Detection

使用 autoencoder 的重建誤差做異常偵測。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

## 1. 建立正常與異常資料

In [ ]:
normal, _ = make_blobs(n_samples=3000, centers=1, n_features=10, cluster_std=1.0, random_state=42)
anomaly = np.random.uniform(low=-8, high=8, size=(180, 10))

X = np.vstack([normal, anomaly])
y = np.hstack([np.zeros(len(normal)), np.ones(len(anomaly))])

x_train_all, x_test, y_train_all, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Autoencoder 只用正常資料訓練
x_train_normal = x_train_all[y_train_all == 0]

scaler = StandardScaler()
x_train_normal = scaler.fit_transform(x_train_normal)
x_test_scaled = scaler.transform(x_test)

print(x_train_normal.shape, x_test_scaled.shape)

## 2. 建立 Autoencoder

In [ ]:
tf.keras.utils.set_random_seed(42)
input_dim = x_train_normal.shape[1]

encoder = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(input_dim,)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(3, activation='relu')
])

decoder = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(3,)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(input_dim)
])

inputs = tf.keras.Input(shape=(input_dim,))
encoded = encoder(inputs)
outputs = decoder(encoded)
autoencoder = tf.keras.Model(inputs, outputs)

autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

## 3. 訓練模型

In [ ]:
history = autoencoder.fit(
    x_train_normal, x_train_normal,
    validation_split=0.2,
    epochs=80,
    batch_size=64,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)]
)

pd.DataFrame(history.history).plot(figsize=(8, 4), title='Reconstruction Loss')
plt.show()

## 4. 設定異常 threshold

In [ ]:
train_recon = autoencoder.predict(x_train_normal, verbose=0)
train_error = np.mean(np.square(x_train_normal - train_recon), axis=1)
threshold = np.percentile(train_error, 95)

print('threshold:', threshold)
plt.hist(train_error, bins=50)
plt.axvline(threshold, color='red', linestyle='--')
plt.title('Train reconstruction error')
plt.show()

## 5. 評估異常偵測

In [ ]:
test_recon = autoencoder.predict(x_test_scaled, verbose=0)
test_error = np.mean(np.square(x_test_scaled - test_recon), axis=1)
y_pred = (test_error > threshold).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['normal', 'anomaly']))

## 6. 套用自己的資料

實務上可先用確認為正常的資料訓練 autoencoder，再用重建誤差 threshold 偵測新資料是否異常。